# Phase 3.4b — Clean up noisy concept-concept relationships

**Buddhist RAG System — Clear Light of Bliss**

## The problem

After Phase 3.3, the graph contains Concept↔Concept relationships with types
like `OF` (71 around `clear_light`), `AND`, `ON` — alongside doctrinally
meaningful types like `DEPENDS_UPON`, `ARISE_FROM`, `FREE_FROM`. Mixed
together, this means "what does Geshe-la teach about X?" returns
doctrinal claims alongside grammatical fragments.

## The plan

`07_semantic_relationships.json` already classifies every relationship as one
of: `PREPOSITION`, `CONJUNCTION`, `RELATIONSHIP_VERB`,
`RELATIONSHIP_ADJPREP`, `TANTRIC_INSTRUCTION`. We act on that classification:

| JSON classification | Action | Graph result |
|---|---|---|
| PREPOSITION | relabel | `CO_OCCURS_WITH` |
| CONJUNCTION | relabel | `CO_OCCURS_WITH` |
| RELATIONSHIP_VERB | keep | (existing doctrinal types) |
| RELATIONSHIP_ADJPREP | keep | (existing doctrinal types) |
| TANTRIC_INSTRUCTION | normalize | `TANTRIC_INSTRUCTION` |

## New Cypher idea introduced here

Neo4j relationship **types are immutable**. You can change properties on a
relationship, but not its type. To "rename" a relationship, the idiom is
**delete the old, create the new in one transaction**:

```cypher
MATCH (a)-[r:OLD_TYPE]->(b)
CREATE (a)-[r2:NEW_TYPE]->(b)
SET r2 = properties(r)
DELETE r
```

Properties are copied so any metadata (the original surface relation, the
source citation) survives the rename.

## Process

1. **Audit** — what edge types exist between Concepts, what properties they carry?
2. **Build classification map** from JSON
3. **Dry-run** — show counts of what would change
4. **Validation gate**
5. **Relabel** in batches
6. **Verify** — re-inventory and confirm

---
## Setup

In [1]:
import json
import os
from pathlib import Path
from collections import defaultdict, Counter
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv()

NEO4J_URI = 'bolt://localhost:7687'
NEO4J_USER = 'neo4j'
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')

SEMANTIC_RELS_PATH = Path('07_semantic_relationships.json')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session() as session:
    n_nodes = session.run('MATCH (n) RETURN count(n) AS c').single()['c']
    n_rels = session.run('MATCH ()-[r]->() RETURN count(r) AS c').single()['c']
    n_mentions = session.run("MATCH ()-[r:MENTIONS]->() RETURN count(r) AS c").single()['c']
    n_concept_rels = session.run(
        'MATCH (:Concept)-[r]-(:Concept) RETURN count(DISTINCT r) AS c'
    ).single()['c']

print(f'Nodes: {n_nodes}')
print(f'Total relationships: {n_rels}')
print(f'  Of which MENTIONS (added in 3.4a): {n_mentions}')
print(f'  Of which Concept↔Concept (our target for 3.4b): {n_concept_rels}')

Nodes: 3533
Total relationships: 5077
  Of which MENTIONS (added in 3.4a): 912
  Of which Concept↔Concept (our target for 3.4b): 683


---
## Step 1 — Audit: what's in the graph?

Two questions:
1. **What relationship types exist between Concepts and how many of each?**
2. **Do those edges have a `relation_type` property** (recording the JSON's
   classification at the time of ingest)? If yes, our job is much simpler —
   we can use that property directly. If no, we need to match edges back to
   the JSON via subject/object/citation.

In [2]:
with driver.session() as session:
    # 1. Inventory of edge types between Concepts
    print('=== Edge types between Concept nodes ===')
    result = session.run('''
        MATCH (:Concept)-[r]->(:Concept)
        RETURN type(r) AS rel_type, count(*) AS count
        ORDER BY count DESC
    ''')
    edge_inventory = [(r['rel_type'], r['count']) for r in result]
    for rt, c in edge_inventory:
        print(f'  {rt:30s} {c:5d}')
    print(f'  {"TOTAL":30s} {sum(c for _, c in edge_inventory):5d}')
    
    # 2. Sample an edge and look at its properties
    print()
    print('=== Sample edges with their properties ===')
    result = session.run('''
        MATCH (a:Concept)-[r]->(b:Concept)
        RETURN a.canonical_form AS subj, type(r) AS rel_type, 
               properties(r) AS props, b.canonical_form AS obj
        LIMIT 5
    ''')
    for row in result:
        print(f'  ({row["subj"]}) -[:{row["rel_type"]} {row["props"]}]-> ({row["obj"]})')

=== Edge types between Concept nodes ===
  OF                               265
  AND                              124
  ON                                32
  DEPENDS_UPON                      27
  MIXING_WITH                       26
  DISSOLVE_WITHIN                   24
  KNOWN_AS                          23
  MEDITATING_ON                     22
  WITH                              16
  FREE_FROM                         15
  EMPTY_OF                          13
  ARISE_FROM                        11
  DISSOLVES_INTO                     9
  FOCUSED_ON                         7
  MOUNTED_UPON                       6
  ARISES_FROM                        6
  EXPLAINED_IN                       6
  ASSOCIATED_WITH                    5
  INSEPARABLE_FROM                   5
  ENGAGING_IN                        5
  DEPENDING_UPON                     5
  RELYING_UPON                       5
  THROUGH                            3
  ATTAINED_IN                        3
  BEFORE               

---
## Step 2 — Build the classification map from JSON

We build a lookup keyed by `(subject, surface_relation, object, citation)`
that returns the classification (`PREPOSITION`, `CONJUNCTION`, etc.) and the
original surface relation. This is our **ground truth** for what each graph
edge should become.

We also surface any duplicates — same `(subject, surface, object, citation)`
appearing twice in the JSON with different classifications would mean our
lookup is ambiguous.

In [3]:
with open(SEMANTIC_RELS_PATH) as f:
    sem = json.load(f)

rels = sem['relationships']

# Build classification lookups
# Key by (subject, object) for fast lookup; value is list of (surface, classification, citation)
json_index = defaultdict(list)
for r in rels:
    key = (r['subject'], r['object'])
    json_index[key].append({
        'surface': r['relation'],
        'classification': r['relation_type'],
        'citation': r['source']['citation'],
    })

# Summary
by_classification = Counter()
for r in rels:
    by_classification[r['relation_type']] += 1

print('=== JSON relationships by classification ===')
for cls, c in by_classification.most_common():
    print(f'  {cls:25s} {c:5d}')
print(f'  {"TOTAL":25s} {len(rels):5d}')
print()

# Sanity: how many distinct (subject, object) pairs are there?
print(f'Distinct (subject, object) pairs in JSON: {len(json_index)}')
print(f'(Some have multiple records because a pair can appear in multiple paragraphs)')
print()

# Show a multi-record example
multi = [(k, v) for k, v in json_index.items() if len(v) > 1]
multi.sort(key=lambda kv: -len(kv[1]))
if multi:
    k, v = multi[0]
    print(f'Example of a pair with {len(v)} records: ({k[0]}) → ({k[1]})')
    for rec in v[:5]:
        print(f'  {rec["surface"]:20s} ({rec["classification"]:20s}) in {rec["citation"]}')
    if len(v) > 5:
        print(f'  ... and {len(v) - 5} more')

=== JSON relationships by classification ===
  PREPOSITION                 311
  RELATIONSHIP_VERB           186
  CONJUNCTION                 125
  RELATIONSHIP_ADJPREP         44
  TANTRIC_INSTRUCTION          17
  TOTAL                       683

Distinct (subject, object) pairs in JSON: 202
(Some have multiple records because a pair can appear in multiple paragraphs)

Example of a pair with 36 records: (mind) → (black_near-attainment)
  of                   (PREPOSITION         ) in CLB.10.§2.p16
  of                   (PREPOSITION         ) in CLB.10.§2.p26
  of                   (PREPOSITION         ) in CLB.10.§2.p26
  of                   (PREPOSITION         ) in CLB.10.§2.p26
  of                   (PREPOSITION         ) in CLB.10.§2.p27
  ... and 31 more


---
## Step 3 — Build the relabel plan

Now we decide, for each graph edge, what its new type should be. Strategy
depends on what Step 1 found:

- **If edges have a `relation_type` property** → use it directly per-edge.
- **If edges only carry the surface form** → look it up in `json_index`.

Either way, we apply the policy:

```
PREPOSITION         → CO_OCCURS_WITH
CONJUNCTION         → CO_OCCURS_WITH
TANTRIC_INSTRUCTION → TANTRIC_INSTRUCTION (unified)
RELATIONSHIP_VERB   → keep existing type
RELATIONSHIP_ADJPREP → keep existing type
```

In [4]:
# Policy mapping: classification → new graph edge type (or None = keep)
RELABEL_POLICY = {
    'PREPOSITION':         'CO_OCCURS_WITH',
    'CONJUNCTION':         'CO_OCCURS_WITH',
    'TANTRIC_INSTRUCTION': 'TANTRIC_INSTRUCTION',
    'RELATIONSHIP_VERB':   None,   # keep existing type
    'RELATIONSHIP_ADJPREP': None,  # keep existing type
}

# Walk the graph, decide what each Concept↔Concept edge should become
with driver.session() as session:
    result = session.run('''
        MATCH (a:Concept)-[r]->(b:Concept)
        RETURN id(r) AS edge_id, 
               a.canonical_form AS subj,
               type(r) AS current_type,
               properties(r) AS props,
               b.canonical_form AS obj
    ''')
    graph_edges = list(result)

print(f'Examining {len(graph_edges)} Concept↔Concept edges in the graph.')
print()

# For each edge, look up the JSON classification
plan = []          # list of (edge_id, current_type, target_type, classification)
unresolved = []    # edges we can't classify
ambiguous = []    # edges where the (subject, object) maps to multiple different classifications

for e in graph_edges:
    key = (e['subj'], e['obj'])
    candidates = json_index.get(key, [])
    
    if not candidates:
        # This edge in the graph doesn't appear in JSON — odd.
        unresolved.append(e)
        continue
    
    # If all candidates have the same classification, easy
    cls_set = {c['classification'] for c in candidates}
    if len(cls_set) == 1:
        cls = next(iter(cls_set))
    else:
        # Multiple classifications for the same (subj, obj). Try to match by surface relation
        # using the edge's current type as a hint.
        current_type_lower = e['current_type'].lower().replace('_', ' ')
        surface_match = [c for c in candidates if c['surface'].lower() == current_type_lower]
        if surface_match:
            cls = surface_match[0]['classification']
        else:
            ambiguous.append((e, candidates))
            continue
    
    target = RELABEL_POLICY.get(cls)
    if target is None:
        # Keep existing type
        plan.append((e['edge_id'], e['current_type'], e['current_type'], cls, False))
    else:
        # Relabel
        will_change = (target != e['current_type'])
        plan.append((e['edge_id'], e['current_type'], target, cls, will_change))

# Summary
to_change = [p for p in plan if p[4]]
to_keep = [p for p in plan if not p[4]]

print(f'  Will relabel:  {len(to_change)}')
print(f'  Will keep:     {len(to_keep)}')
print(f'  Unresolved (no JSON match):  {len(unresolved)}')
print(f'  Ambiguous (need disambiguation):  {len(ambiguous)}')
print()

# Per-classification breakdown of the plan
print('=== Plan by classification ===')
cls_counter = Counter(p[3] for p in plan)
for cls, c in cls_counter.most_common():
    target = RELABEL_POLICY.get(cls)
    print(f'  {cls:25s} {c:5d} → {target if target else "(keep existing type)"}')

if unresolved:
    print()
    print('=== Unresolved edges (no JSON match) — first 5 ===')
    for e in unresolved[:5]:
        print(f'  ({e["subj"]}) -[:{e["current_type"]}]-> ({e["obj"]})')

if ambiguous:
    print()
    print('=== Ambiguous edges — first 3 ===')
    for e, cands in ambiguous[:3]:
        print(f'  ({e["subj"]}) -[:{e["current_type"]}]-> ({e["obj"]}) — JSON candidates:')
        for c in cands:
            print(f'      {c["surface"]:15s} {c["classification"]}')

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=3, column=16, offset=59>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 59, 'line': 3, 'column': 16}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        MATCH (a:Concept)-[r]->(b:Concept)\n        RETURN id(r) AS edge_id, \n               a.canonical_form AS subj,\n               type(r) AS current_type,\n               properties(r) AS props,\n               b.canonical_form AS obj\n    '


Examining 683 Concept↔Concept edges in the graph.

  Will relabel:  453
  Will keep:     230
  Unresolved (no JSON match):  0
  Ambiguous (need disambiguation):  0

=== Plan by classification ===
  PREPOSITION                 313 → CO_OCCURS_WITH
  RELATIONSHIP_VERB           186 → (keep existing type)
  CONJUNCTION                 125 → CO_OCCURS_WITH
  RELATIONSHIP_ADJPREP         44 → (keep existing type)
  TANTRIC_INSTRUCTION          15 → TANTRIC_INSTRUCTION


---
## Step 4 — Validation gate

Three checks before writing:

1. **Plan accounts for all edges**: `len(plan) + len(unresolved) + len(ambiguous)` equals total graph edges.
2. **No catastrophic relabeling**: doctrinal verbs (`DEPENDS_UPON`, `ARISE_FROM`, etc.) must NOT appear in the "will relabel" set.
3. **The change set is sensible**: relabels are mostly `OF`, `AND`, `ON` going to `CO_OCCURS_WITH`.

In [5]:
# Check 1
accounted = len(plan) + len(unresolved) + len(ambiguous)
print(f'Check 1: accounted-for edges = {accounted}, graph total = {len(graph_edges)}')
check1 = (accounted == len(graph_edges))
print(f'  {"✓" if check1 else "✗"} {"all accounted for" if check1 else "MISMATCH"}')

# Check 2: no doctrinal verb in the relabel set
DOCTRINAL_PREFIXES = ('DEPENDS', 'ARISE', 'ARISES', 'DISSOLVE', 'DISSOLVES',
                     'FREE_FROM', 'EMPTY_OF', 'KNOWN_AS', 'MEDITATING',
                     'MIXING', 'EXPLAINED', 'MOUNTED', 'INSEPARABLE',
                     'FOCUSED', 'ACCUSTOMED', 'INDICATIVE')
doctrinal_in_relabel = [p for p in to_change if any(
    p[1].startswith(prefix) for prefix in DOCTRINAL_PREFIXES
)]
print(f'Check 2: doctrinal types in relabel set = {len(doctrinal_in_relabel)}')
check2 = (len(doctrinal_in_relabel) == 0)
print(f'  {"✓" if check2 else "✗"} {"no doctrinal types being relabeled" if check2 else "DOCTRINAL TYPES WOULD BE RELABELED — STOP"}')
if doctrinal_in_relabel:
    for p in doctrinal_in_relabel[:5]:
        print(f'    {p[1]} → {p[2]}')

# Check 3: what types ARE being relabeled?
print(f'Check 3: source types being relabeled (top 10):')
relabel_sources = Counter(p[1] for p in to_change)
for t, c in relabel_sources.most_common(10):
    print(f'    {t:25s} {c:5d}')

VALIDATION_PASSED = check1 and check2
print()
if VALIDATION_PASSED:
    print('✓ All checks passed. Safe to proceed to Step 5.')
else:
    print('✗ Validation failed. Do not run Step 5.')

Check 1: accounted-for edges = 683, graph total = 683
  ✓ all accounted for
Check 2: doctrinal types in relabel set = 0
  ✓ no doctrinal types being relabeled
Check 3: source types being relabeled (top 10):
    OF                          265
    AND                         124
    ON                           32
    WITH                         16
    THROUGH                       3
    BEFORE                        3
    FOR                           2
    DURING                        2
    FROM                          2
    WITHIN                        2

✓ All checks passed. Safe to proceed to Step 5.


---
## Step 5 — Relabel

For each edge that needs to change, we delete the old typed relationship and
create a new one with the target type, copying properties across. We track the
old surface form on the new edge as `original_relation` so we don't lose that
information.

The Cypher pattern:

```cypher
MATCH (a)-[r]->(b)
WHERE id(r) = $edge_id
CREATE (a)-[r2:CO_OCCURS_WITH]->(b)
SET r2 = properties(r), r2.original_type = type(r)
DELETE r
```

Run in batches with `UNWIND`. Because Cypher doesn't allow the target
relationship type to come from a parameter, we run a separate batch per target
type (e.g. one batch for `CO_OCCURS_WITH`, one for `TANTRIC_INSTRUCTION`).

In [6]:
if not VALIDATION_PASSED:
    raise RuntimeError('Validation failed in Step 4 — refusing to write.')

# Group changes by target type
by_target = defaultdict(list)
for edge_id, src_type, tgt_type, cls, will_change in to_change:
    by_target[tgt_type].append({'edge_id': edge_id, 'src_type': src_type})

print(f'Relabeling {len(to_change)} edges across {len(by_target)} target types')
print()

BATCH_SIZE = 200

for target_type, edges in by_target.items():
    # Build the query string — target type is interpolated since it can't be a parameter
    query = f'''
        UNWIND $edges AS e
        MATCH (a)-[r]->(b)
        WHERE id(r) = e.edge_id
        CREATE (a)-[r2:{target_type}]->(b)
        SET r2 = properties(r), r2.original_type = e.src_type
        DELETE r
        RETURN count(r2) AS created
    '''
    written = 0
    for i in range(0, len(edges), BATCH_SIZE):
        batch = edges[i:i + BATCH_SIZE]
        with driver.session() as session:
            result = session.execute_write(
                lambda tx: tx.run(query, edges=batch).single()
            )
        written += result['created']
    print(f'  {target_type:25s} {written:5d} edges relabeled')

# Cross-check
with driver.session() as session:
    actual_co = session.run(
        'MATCH ()-[r:CO_OCCURS_WITH]->() RETURN count(r) AS c'
    ).single()['c']
    actual_ti = session.run(
        'MATCH ()-[r:TANTRIC_INSTRUCTION]->() RETURN count(r) AS c'
    ).single()['c']

print()
print(f'CO_OCCURS_WITH in graph now: {actual_co}')
print(f'TANTRIC_INSTRUCTION in graph now: {actual_ti}')

Relabeling 453 edges across 2 target types



Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. id is deprecated. It is replaced by elementId or consider using an application-generated id.', position=<SummaryInputPosition line=4, column=15, offset=69>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 69, 'line': 4, 'column': 15}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n        UNWIND $edges AS e\n        MATCH (a)-[r]->(b)\n        WHERE id(r) = e.edge_id\n        CREATE (a)-[r2:CO_OCCURS_WITH]->(b)\n        SET r2 = properties(r), r2.original_type = e.src_type\n        DELETE r\n        RETURN count(r2) AS created\n    '
Received notification from DBMS server: <GqlStatusObje

  CO_OCCURS_WITH              438 edges relabeled
  TANTRIC_INSTRUCTION          15 edges relabeled

CO_OCCURS_WITH in graph now: 438
TANTRIC_INSTRUCTION in graph now: 15


---
## Step 6 — Verify

Re-inventory edge types between Concepts. We expect:

- A new `CO_OCCURS_WITH` type with hundreds of edges (from `OF` + `AND` + `ON` + others)
- A new `TANTRIC_INSTRUCTION` type with ~17 edges
- Doctrinal types (`DEPENDS_UPON`, `ARISE_FROM`, etc.) unchanged in count
- No more `OF`, `AND`, or `ON` types

In [7]:
with driver.session() as session:
    result = session.run('''
        MATCH (:Concept)-[r]->(:Concept)
        RETURN type(r) AS rel_type, count(*) AS count
        ORDER BY count DESC
    ''')
    new_inventory = [(r['rel_type'], r['count']) for r in result]

print('=== Concept↔Concept edge types AFTER cleanup ===')
for rt, c in new_inventory:
    print(f'  {rt:30s} {c:5d}')
print(f'  {"TOTAL":30s} {sum(c for _, c in new_inventory):5d}')

# Spot-check: doctrinal types unchanged
before_map = dict(edge_inventory)
after_map = dict(new_inventory)

print()
print('=== Spot-check: doctrinal types unchanged ===')
doctrinal_types = [t for t in before_map if any(t.startswith(p) for p in DOCTRINAL_PREFIXES)]
for t in sorted(doctrinal_types):
    b = before_map.get(t, 0)
    a = after_map.get(t, 0)
    marker = '✓' if a == b else '✗'
    print(f'  {marker} {t:25s} before={b:4d}  after={a:4d}')

# Spot-check: grammatical types gone
print()
print('=== Spot-check: grammatical types removed ===')
for t in ('OF', 'AND', 'ON', 'WITH', 'OR'):
    a = after_map.get(t, 0)
    marker = '✓' if a == 0 else '✗'
    print(f'  {marker} {t}: {a} remaining ({"removed" if a == 0 else "STILL PRESENT"})')

=== Concept↔Concept edge types AFTER cleanup ===
  CO_OCCURS_WITH                   438
  DEPENDS_UPON                      27
  MIXING_WITH                       26
  DISSOLVE_WITHIN                   24
  KNOWN_AS                          23
  MEDITATING_ON                     22
  FREE_FROM                         15
  TANTRIC_INSTRUCTION               15
  EMPTY_OF                          13
  ARISE_FROM                        11
  DISSOLVES_INTO                     9
  FOCUSED_ON                         7
  MOUNTED_UPON                       6
  ARISES_FROM                        6
  EXPLAINED_IN                       6
  ASSOCIATED_WITH                    5
  INSEPARABLE_FROM                   5
  ENGAGING_IN                        5
  DEPENDING_UPON                     5
  RELYING_UPON                       5
  ATTAINED_IN                        3
  INDICATIVE_OF                      2
  INCLUDED_WITHIN                    2
  ACCUSTOMED_TO                      2
  DISSOLVING_IN

---
## Step 7 — A query that wasn't possible before

Now we can ask "what does Geshe-la teach about the relationship between
`clear_light` and other concepts?" — filtering to doctrinal claims only,
excluding the co-occurrence noise.

In [8]:
with driver.session() as session:
    print('=== Doctrinal claims involving clear_light ===')
    result = session.run('''
        MATCH (c:Concept {canonical_form: 'clear_light'})-[r]-(other:Concept)
        WHERE NOT type(r) IN ['CO_OCCURS_WITH', 'TANTRIC_INSTRUCTION']
        RETURN type(r) AS relation, other.canonical_form AS other_concept,
               r.original_type AS original
        ORDER BY type(r), other.canonical_form
    ''')
    for row in result:
        print(f'  clear_light --[{row["relation"]}]-- {row["other_concept"]}')

    print()
    print('=== Co-occurrences of clear_light (sample) ===')
    result = session.run('''
        MATCH (c:Concept {canonical_form: 'clear_light'})-[r:CO_OCCURS_WITH]-(other:Concept)
        RETURN other.canonical_form AS other_concept, count(r) AS edge_count
        ORDER BY edge_count DESC LIMIT 10
    ''')
    for row in result:
        print(f'  clear_light ↔ {row["other_concept"]:25s} ({row["edge_count"]} co-occurrences)')

=== Doctrinal claims involving clear_light ===
  clear_light --[ARISE_FROM]-- clear_light
  clear_light --[ARISE_FROM]-- ordinary_beings
  clear_light --[ARISE_FROM]-- union
  clear_light --[ATTAINED_IN]-- clear_light
  clear_light --[DEPENDING_UPON]-- illusory_body
  clear_light --[DEPENDS_UPON]-- central_channel
  clear_light --[DEPENDS_UPON]-- central_channel
  clear_light --[DEPENDS_UPON]-- completion_stage
  clear_light --[DEPENDS_UPON]-- illusory_body
  clear_light --[DEPENDS_UPON]-- illusory_body
  clear_light --[DEPENDS_UPON]-- intermediate_state
  clear_light --[DEPENDS_UPON]-- intermediate_state
  clear_light --[DEPENDS_UPON]-- isolated_mind
  clear_light --[DEPENDS_UPON]-- isolated_mind
  clear_light --[DEPENDS_UPON]-- union
  clear_light --[DISSOLVE_WITHIN]-- central_channel
  clear_light --[DISSOLVE_WITHIN]-- central_channel
  clear_light --[DISSOLVING_INTO]-- indestructible_drop
  clear_light --[EMPTY_OF]-- mind
  clear_light --[ENGAGING_IN]-- meditation
  clear_light --[

In [9]:
driver.close()
print('Neo4j connection closed.')

Neo4j connection closed.


---
## Done

The concept-relationship vocabulary is now clean:

- **Doctrinal verbs** (`DEPENDS_UPON`, `ARISE_FROM`, `FREE_FROM`, etc.) preserved as-is
- **Grammatical noise** (`OF`, `AND`, `ON`) consolidated as `CO_OCCURS_WITH`
- **Tantric instructions** unified under `TANTRIC_INSTRUCTION`
- **Original type preserved** on every relabeled edge via `original_type` property,
  so no information is lost — you can always recover the original surface form.

**KI-002 resolved.** Phase 3.4 complete. Next: Phase 4 — hybrid retrieval.

Update the guiding doc to:
- Move KI-002 to RESOLVED (link to this notebook)
- Move KI-003 to RESOLVED (link to phase3_4a notebook)
- Check off Phase 3.4a and 3.4b in the Complete Checklist